In [45]:
import numpy as np
import pandas as pd

from scipy.sparse import hstack
from category_encoders import BinaryEncoder
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer

In [46]:
# Global Variables
FILE_PATH: str = "https://raw.githubusercontent.com/vinayak-ensemble/Dataset-TV-Shows-OTT/refs/heads/main/tv-shows.csv" 
OTT_DF: pd.DataFrame = pd.read_csv(FILE_PATH)
OTT_DF.set_index('id', inplace=True)

## 1. Pre-processing

In [47]:
# Converting the date_added to datetime format:
OTT_DF['date_added'] = pd.to_datetime(OTT_DF['date_added'].str.strip(), format='%B %d, %Y')

In [48]:
OTT_DF.columns

Index(['type', 'title', 'director', 'cast', 'country', 'date_added',
       'release_year', 'rating', 'duration', 'listed_in', 'description',
       'platform'],
      dtype='str')

In [49]:
# Handle missing values:

# -- High missing value columns --

# Treatment: 
# - need to fill in `Unknown` for the missing values in these cases.
# - since director has a 30% missing value we need to check if it's fit to be modeled on. 

treatment_cols = ['director', 'cast', 'country']
OTT_DF[treatment_cols] = OTT_DF[treatment_cols].fillna('Unknown')
OTT_DF[treatment_cols].isnull().sum()

director    0
cast        0
country     0
dtype: int64

In [50]:
# Fixing the duration column by brining in values from rating
mask = (
    OTT_DF['duration'].isna() &
    OTT_DF['rating'].astype(str).str.contains(r'^\d+\s*min$', na=False)
)

OTT_DF.loc[mask, 'duration'] = OTT_DF.loc[mask, 'rating']
OTT_DF.loc[mask, 'rating'] = np.nan

In [51]:
# Handle missing values:

# -- Low missing value columns --

# Treatment: 
# - need to fill in with the mean of the numeric columns.
# - need to fill in with the mode of the string columns. 

mean_treat_cols = ['date_added' ,'release_year']
mode_tread_grp_cols = ['rating']

# fillna with mean of the columns
for col in mean_treat_cols:
    OTT_DF[col] = OTT_DF[col].fillna(OTT_DF[col].mean())

# fillna with the mode of the column by type
duration_mode_by_type = (
    OTT_DF.groupby('type')[mode_tread_grp_cols[0]]
    .agg(lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
)

OTT_DF[mode_tread_grp_cols[0]] = OTT_DF[mode_tread_grp_cols[0]].fillna(
    OTT_DF['type'].map(duration_mode_by_type)
)

OTT_DF[mean_treat_cols + mode_tread_grp_cols].isnull().sum()

date_added      0
release_year    0
rating          0
dtype: int64

## 2. Data Transformation

In [52]:
# Transform the duration column into 2 seperate columns based on type.

# extract numeric value
OTT_DF['duration_num'] = OTT_DF['duration'].str.extract(r'(\d+)').astype(float)

# create separate features
OTT_DF['duration_movies'] = np.where(
    OTT_DF['type'] == 'Movie',
    OTT_DF['duration_num'],
    np.nan
)

OTT_DF['duration_tv_series'] = np.where(
    OTT_DF['type'] == 'TV Show',
    OTT_DF['duration_num'],
    np.nan
)

# optional: fill missing values with 0
OTT_DF[['duration_movies', 'duration_tv_series']] = (
    OTT_DF[['duration_movies', 'duration_tv_series']]
    .fillna(0)
)

# drop old columns
OTT_DF.drop(columns=['duration', 'duration_num'], inplace=True)
OTT_DF[['duration_movies', 'duration_tv_series']].head()

,duration_movies,duration_tv_series
id,,
1,61.0,0.0
2,123.0,0.0
3,0.0,1.0
4,0.0,1.0
5,63.0,0.0


In [53]:
# Transform the data to create meaningful bins

cols_to_bin = ['duration_movies', 'duration_tv_series', 'date_added', 'release_year']

# updating the column to only have the year so that we can bin it
OTT_DF['date_added'] = OTT_DF['date_added'].dt.year

# using quartile bins to have more or less equal number of values in each bin
for col in cols_to_bin:
    OTT_DF[f'{col}_bin'] = pd.qcut(
        OTT_DF[col],
        q=6,
        labels=False,
        duplicates='drop'
    )

OTT_DF.drop(
    columns=cols_to_bin,
    inplace=True
)

In [54]:
for col in cols_to_bin:
    print(OTT_DF[f'{col}_bin'].value_counts(), end='\n\n')

duration_movies_bin
0    3120
2    1639
1    1583
4    1548
3    1448
Name: count, dtype: int64

duration_tv_series_bin
0    8357
1     981
Name: count, dtype: int64

date_added_bin
1    4055
2    2030
3    1678
0    1575
Name: count, dtype: int64

release_year_bin
2    2414
4    1997
0    1678
1    1498
3    1086
5     665
Name: count, dtype: int64



In [55]:
# binary encodding the rating column as having 17 columns through one-hot encodding will be a lot.
be = BinaryEncoder(cols=['rating'])
OTT_DF = be.fit_transform(OTT_DF)

# label encoding the values for platform: [manual label encodding]
label_encoder_platform = {'Disney': 0, 'Netflix' : 1}
OTT_DF['platform'] = (OTT_DF['platform'].replace(label_encoder_platform)).astype(int)

label_encoder_type = {'Movie': 0, 'TV Show' : 1}
OTT_DF['type'] = (OTT_DF['type'].replace(label_encoder_type)).astype(int)
OTT_DF.head()

,type,title,director,cast,country,rating_0,rating_1,rating_2,rating_3,listed_in,description,platform,duration_movies_bin,duration_tv_series_bin,date_added_bin,release_year_bin
id,,,,,,,,,,,,,,,,
1,0,ChuChuTV Surprise Eggs Learning Videos (English),Unknown,Unknown,Unknown,0,0,0,1,Children & Family Movies,From colors and letters to animals of all kind...,1,1,0,1,4
2,0,The Journey Is the Destination,Bronwen Hughes,"Ben Schnetzer, Kelly Macdonald, Sam Hazeldine,...",United States,0,0,1,0,Dramas,Spirited 22-year-old activist and photojournal...,1,4,0,0,2
3,1,Champions,Unknown,"Anders Holm, Fortune Feimster, Andy Favreau, J...",United States,0,0,1,1,TV Comedies,"Years after getting his girlfriend pregnant, w...",1,0,0,3,3
4,1,The Returned,Unknown,"Anne Cosigny, Frédéric Pierrot, Clotilde Hesme...",France,0,1,0,0,"International TV Shows, TV Dramas, TV Horror",On returning home and finding they're believed...,1,0,0,1,2
5,0,Super Bheem Bana Vajraveer,Sumit Das,"Sonal Kaushal, Rupa Bhimani, Julie Tejwani, Sa...",India,0,1,0,1,Children & Family Movies,"Hoping to find a magical root, a monster has c...",1,1,0,1,3


In [56]:
OTT_DF.info()

<class 'pandas.DataFrame'>
RangeIndex: 9338 entries, 1 to 9338
Data columns (total 16 columns):
 #   Column                  Non-Null Count  Dtype
---  ------                  --------------  -----
 0   type                    9338 non-null   int64
 1   title                   9338 non-null   str  
 2   director                9338 non-null   str  
 3   cast                    9338 non-null   str  
 4   country                 9338 non-null   str  
 5   rating_0                9338 non-null   int64
 6   rating_1                9338 non-null   int64
 7   rating_2                9338 non-null   int64
 8   rating_3                9338 non-null   int64
 9   listed_in               9338 non-null   str  
 10  description             9338 non-null   str  
 11  platform                9338 non-null   int64
 12  duration_movies_bin     9338 non-null   int64
 13  duration_tv_series_bin  9338 non-null   int64
 14  date_added_bin          9338 non-null   int64
 15  release_year_bin        9338 non

In [57]:
def tfidf_svd_features(df, text_cols, variance_threshold=0.85):
    tfidf_matrices = []

    for col in text_cols:
        vectorizer = TfidfVectorizer(
            stop_words='english',
            max_features=1000
        )

        X = vectorizer.fit_transform(
            df[col].fillna('').astype(str)
        )

        tfidf_matrices.append(X)

    X_text = hstack(tfidf_matrices)

    max_components = min(X_text.shape) - 1

    svd = TruncatedSVD(
        n_components=max_components,
        random_state=42
    )

    X_svd = svd.fit_transform(X_text)

    cum_var = np.cumsum(svd.explained_variance_ratio_)
    n_components = np.argmax(cum_var >= variance_threshold) + 1

    X_svd = X_svd[:, :n_components]

    svd_df = pd.DataFrame(
        X_svd,
        columns=[f'text_svd_{i}' for i in range(n_components)],
        index=df.index
    )

    return svd_df

# -----------------------------------------------------------------
cat_cols = ['country', 'title', 'director', 'cast', 'description']

text_features = tfidf_svd_features(
    OTT_DF,
    cat_cols,
    variance_threshold=0.85
)

OTT_DF = pd.concat([OTT_DF, text_features], axis=1)
OTT_DF.drop(cat_cols, axis=1, inplace=True)

## 3. Modelling

In [64]:
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.multioutput import MultiOutputClassifier
from sklearn.multiclass import OneVsRestClassifier

from sklearn.linear_model import LogisticRegression
from lightgbm import LGBMClassifier

from sklearn.metrics import (
    f1_score,
    accuracy_score,
    hamming_loss
)

from iterstrat.ml_stratifiers import MultilabelStratifiedKFold
from pprint import pprint

In [42]:
# Prepare training data

df = OTT_DF.copy()

# Convert genre strings to list of labels
y_raw = (
    df["listed_in"]
    .astype(str)
    .str.split(",")
    .apply(lambda x: [i.strip() for i in x])
)

# Multi-label encoding
mlb = MultiLabelBinarizer()
Y = mlb.fit_transform(y_raw)

# Features
X = df.drop(columns=["listed_in"])

print(f"Shape of X: {X.shape}")
print(f"Shape of Y: {Y.shape}")
print(f"Number of genres: {len(mlb.classes_)}")

Shape of X: (9338, 1704)
Shape of Y: (9338, 84)
Number of genres: 84


In [ ]:
# cross-valiation object for multilable classification

cv = MultilabelStratifiedKFold(
    n_splits=2,
    shuffle=True,
    random_state=42
)


In [ ]:
# -- Logistic Regression --
lr_model = OneVsRestClassifier(
    LogisticRegression(
        solver="liblinear",      # Good for binary classification in One-vs-Rest
        C=1.0,
        max_iter=1000,
        class_weight="balanced", # Helps with imbalanced genres
        random_state=42
    ),
    n_jobs=-2
)

lr_scores = []

print("\nRunning Logistic Regression CV...\n")

for fold, (train_idx, test_idx) in enumerate(cv.split(X, Y), start=1):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    Y_train = Y[train_idx]
    Y_test = Y[test_idx]

    lr_model.fit(X_train, Y_train)

    Y_pred = lr_model.predict(X_test)

    metrics = {
        "Fold": fold,
        "Micro_F1": round(f1_score(Y_test, Y_pred, average="micro", zero_division=0), 6),
        "Macro_F1": round(f1_score(Y_test, Y_pred, average="macro", zero_division=0), 6),
        "Subset_Accuracy": round(accuracy_score(Y_test, Y_pred), 6),
        "Hamming_Loss": round(hamming_loss(Y_test, Y_pred), 6)
    }

    lr_scores.append(metrics)

    pprint(metrics)

lr_results = pd.DataFrame(lr_scores)

print("\nFold-wise Results:")
print(lr_results)

print("\nAverage Logistic Regression Metrics:")
print(lr_results.mean(numeric_only=True))


Running Logistic Regression CV...

{'Fold': 1, 'Micro_F1': 0.5995952997976499, 'Macro_F1': 0.3574944176426099, 'Subset_Accuracy': 0.10138050043140638, 'Hamming_Loss': 0.028963289370968405}
{'Fold': 2, 'Micro_F1': 0.5901284557386205, 'Macro_F1': 0.35698519867751927, 'Subset_Accuracy': 0.09442790301999149, 'Hamming_Loss': 0.029728990703044297}

Fold-wise Results:
   Fold  Micro_F1  Macro_F1  Subset_Accuracy  Hamming_Loss
0     1  0.599595  0.357494         0.101381      0.028963
1     2  0.590128  0.356985         0.094428      0.029729

Average Logistic Regression Metrics:
Fold               1.500000
Micro_F1           0.594862
Macro_F1           0.357240
Subset_Accuracy    0.097904
Hamming_Loss       0.029346
dtype: float64


In [ ]:
# --- LightGBM ---

lgbm_model = OneVsRestClassifier(
    LGBMClassifier(
        device="gpu",
        gpu_platform_id=0,
        gpu_device_id=0,
        boosting_type="gbdt",
        objective="binary",
        n_estimators=100,
        learning_rate=0.1,
        num_leaves=63,
        max_depth=-1,
        subsample=0.8,
        subsample_freq=1,
        colsample_bytree=0.8,
        reg_alpha=0.1,
        reg_lambda=0.1,
        random_state=42,
        verbosity=-1
    ),
    n_jobs=-2
)
lgbm_scores = []

print("\nRunning LightGBM CV...\n")

for fold, (train_idx, test_idx) in enumerate(cv.split(X, Y), start=1):

    X_train = X.iloc[train_idx]
    X_test = X.iloc[test_idx]

    Y_train = Y[train_idx]
    Y_test = Y[test_idx]

    lgbm_model.fit(X_train, Y_train)

    Y_pred = lgbm_model.predict(X_test)

    metrics = {
        "Fold": fold,
        "Micro_F1": round(f1_score(Y_test, Y_pred, average="micro", zero_division=0), 6),
        "Macro_F1": round(f1_score(Y_test, Y_pred, average="macro", zero_division=0), 6),
        "Subset_Accuracy": round(accuracy_score(Y_test, Y_pred), 6),
        "Hamming_Loss": round(hamming_loss(Y_test, Y_pred), 6)
    }
    
    lgbm_scores.append(metrics)

    pprint(metrics)

lgbm_results = pd.DataFrame(lgbm_scores)

print(lgbm_results)

print("\nAverage LightGBM Metrics")
print(lgbm_results.mean(numeric_only=True))



Running LightGBM CV...

{'Fold': 1,
 'Hamming_Loss': 0.019357,
 'Macro_F1': 0.16568,
 'Micro_F1': 0.507899,
 'Subset_Accuracy': 0.130285}
{'Fold': 2,
 'Hamming_Loss': 0.019478,
 'Macro_F1': 0.167424,
 'Micro_F1': 0.507963,
 'Subset_Accuracy': 0.139302}
   Fold  Micro_F1  Macro_F1  Subset_Accuracy  Hamming_Loss
0     1  0.507899  0.165680         0.130285      0.019357
1     2  0.507963  0.167424         0.139302      0.019478

Average LightGBM Metrics
Fold               1.500000
Micro_F1           0.507931
Macro_F1           0.166552
Subset_Accuracy    0.134794
Hamming_Loss       0.019417
dtype: float64


In [67]:
# Final Comparison

comparison = pd.DataFrame({
    "LogisticRegression": lr_results.mean(numeric_only=True),
    "LightGBM": lgbm_results.mean(numeric_only=True)
}).T

print(comparison.sort_values("Micro_F1", ascending=False))

                    Fold  Micro_F1  Macro_F1  Subset_Accuracy  Hamming_Loss
LogisticRegression   1.5  0.594862  0.357240         0.097904      0.029346
LightGBM             1.5  0.507931  0.166552         0.134794      0.019417
